# Relaxometry of a spin-1/2
This notebook simulates the time evolution of the spin under the influence of a Relaxometry experiment.
It covers:
- Static Spin Hamiltonian
- Relaxometry simulation
- Influence of pure dephasing time $T_\phi$

In [ ]:
# Imports
import matplotlib.pyplot as plt
import numpy as np
import scipy.constants as const
from qutip import (
    tensor, qeye, sigmax, sigmay, sigmaz,
    basis, mesolve, sigmam, Qobj, Bloch
)

from spinfinity import ESR, lind, CC, SpinSys, bath, hamil
from spinfinity import helperfunctions as hf

%load_ext autoreload
%autoreload 2

## Static spin Hamiltonian
The static spin Hamiltonian includes an external field in z-direction which defines the quantization axis

In [ ]:
spin = SpinSys([0.5])
spin.gfactorvector = [2.0]
spin.B = [0.0, 0.0, 0.5]  # T
spin.BTip = [1e-9, 0.0, 0.0]  # T

hamil.calc_EigEnergies(spin, rebuildH=True)
f_e= float(spin.E_All_inGHz[1] - spin.E_All_inGHz[0])  # GHz

hamil.plot_ZeemanDiagramm(spin, energyscale='GHz');

## Continous-wave frequency sweep
Simulating a ESR frequency sweep also reveals the resonance frequency $f_e$. 

In [ ]:
ESR.plot_FreqSweep(spin, freq_array = np.linspace(12,16,1000), lw=0.05);

## Constructing collapse operators
We construct $T_1$ collapse operators which determine the relaxometry signal and $T_phi$ collapse operators which have no effect on the relaxometry signal

In [ ]:
# Tunneling Parameters
spin.T = 0.2 # K
spin.V_DC = 0.0  # mV
spin.TipPolarization = [0, 0, 0.5]
spin.U = np.array([0.0])
spin.G_st = 0.1e-9  # S
spin.G_ss = [1e-9]  # S
spin.G_tt = 1 # Allow for tip-tip scattering
spin.b0 = 0.0

# Relaxation
bath.calc_Rates(spin)
bath.calc_Lifetimes(spin)
T1 = (1/spin.T_state_tot[0]+1/spin.T_state_tot[1])**-1*1e9

c_ops_T1 = lind.calc_collapseOperators(spin, threshold_percent=1e-9, recalcRates=True)


# Decoherence
T_phi = 200  # ns
c_op_Tphi = lind.calc_dephasingOperator(spin, spin_idx=0, T_phi=T_phi)
spin.collapse_ops = c_ops_T1 + [c_op_Tphi]


## Relaxomtery Simulation

In [ ]:
# Parameters and initial state
rho_therm = lind.initialize_thermal(spin)

spin.f_RF = f_e # GHz
spin.RabiRate = 50e-3 # GHz

tau_array = np.linspace(0, 5*T1, 500)  # ns
_, result_relax = CC.calc_Relaxometry(spin, tau_array, e_ops=None)

# Perform an exponential Fit
x = result_relax.times
y = result_relax.expect[2]
x_fit, y_fit, popt, perr = hf.fit_exponential(x, y)
A_fit, T_fit, C_fit = popt

# Plot the result
plt.figure(figsize=(7, 4))
plt.plot(x, y, "o", ms=3, label="Simulated trace")
plt.plot(x_fit, y_fit, "-", lw=2, label="Exponential fit")
plt.xlabel(r"Free evolution time $\tau$ (ns)")
plt.ylabel(r"Final $\langle S_z \rangle$")
plt.title(f"Fit: A={A_fit:.4f}, T={T_fit:.2f} ns, C={C_fit:.4f}")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

print(f"T1 time from tunnel rates: {T1:.2f} ns")

### Influence of $T_\phi$
Simulating the signal with $T_\phi=50~ns$ produces the same result.

In [ ]:
# Decoherence
T_phi = 50  # ns
c_op_Tphi = lind.calc_dephasingOperator(spin, spin_idx=0, T_phi=T_phi)
spin.collapse_ops = c_ops_T1 + [c_op_Tphi]

# Parameters and initial state
rho_therm = lind.initialize_thermal(spin)

spin.f_RF = f_e # GHz
spin.RabiRate = 50e-3 # GHz

tau_array = np.linspace(0, 5*T1, 500)  # ns
_, result_relax = CC.calc_Relaxometry(spin, tau_array, e_ops=None)

# Perform an exponential Fit
x = result_relax.times
y = result_relax.expect[2]
x_fit, y_fit, popt, perr = hf.fit_exponential(x, y)
A_fit, T_fit, C_fit = popt

# Plot the result
plt.figure(figsize=(7, 4))
plt.plot(x, y, "o", ms=3, label="Simulated trace")
plt.plot(x_fit, y_fit, "-", lw=2, label="Exponential fit")
plt.xlabel(r"Free evolution time $\tau$ (ns)")
plt.ylabel(r"Final $\langle S_z \rangle$")
plt.title(f"Fit: A={A_fit:.4f}, T={T_fit:.2f} ns, C={C_fit:.4f}")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

print(f"T1 time from tunnel rates: {T1:.2f} ns")